In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR    = Path("../..") / "output" / "evr" / "01-raw"
OUTPUT_DIR = Path("../..") / "output" / "evr" / "02-silver"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_flujo = pd.read_parquet(RAW_DIR / "raw_flujo_interprovincial.parquet")
df_saldo = pd.read_parquet(RAW_DIR / "raw_saldo_interprovincial.parquet")
df_edad  = pd.read_parquet(RAW_DIR / "raw_flujo_nacional_edad_sexo.parquet")

print(f"✅ flujo_interprovincial → {df_flujo.shape[0]:>7} filas | {df_flujo.shape[1]} cols")
print(f"✅ saldo_interprovincial → {df_saldo.shape[0]:>7} filas | {df_saldo.shape[1]} cols")
print(f"✅ flujo_edad_sexo       → {df_edad.shape[0]:>7} filas | {df_edad.shape[1]} cols")


✅ flujo_interprovincial →  115148 filas | 6 cols
✅ saldo_interprovincial →  200741 filas | 6 cols
✅ flujo_edad_sexo       →    3864 filas | 5 cols


In [2]:
TEXTO_FIJO = 'Flujo de migraciones interprovinciales'

antes = len(df_flujo)
df_flujo_clean = df_flujo[
    (df_flujo['provincia_origen']  != 'Total Nacional') &
    (df_flujo['provincia_origen']  != TEXTO_FIJO) &
    (df_flujo['provincia_destino'] != TEXTO_FIJO) &
    df_flujo['provincia_origen'].notna() &
    df_flujo['provincia_destino'].notna() &
    df_flujo['valor'].notna()
].copy()

print(f"  ⚠️  Filas eliminadas (totales/texto fijo): {antes - len(df_flujo_clean)}")
print(f"  Flujo limpio: {df_flujo_clean.shape}")

df_flujo_total = df_flujo_clean[df_flujo_clean['sexo'] == 'Total'].copy()
print(f"  Flujo Total (sexo agregado): {df_flujo_total.shape}")
print(f"  Provincias origen:  {df_flujo_total['provincia_origen'].nunique()}")
print(f"  Provincias destino: {df_flujo_total['provincia_destino'].nunique()}")
print(f"  Años: {sorted(df_flujo_total['anyo'].unique().tolist())}")


  ⚠️  Filas eliminadas (totales/texto fijo): 112964
  Flujo limpio: (2184, 6)
  Flujo Total (sexo agregado): (728, 6)
  Provincias origen:  52
  Provincias destino: 1
  Años: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]


In [3]:
salidas = (
    df_flujo_total
    .groupby(['provincia_origen', 'anyo'])['valor']
    .sum()
    .reset_index()
    .rename(columns={'provincia_origen': 'provincia', 'valor': 'salidas'})
)

entradas = (
    df_flujo_total
    .groupby(['provincia_destino', 'anyo'])['valor']
    .sum()
    .reset_index()
    .rename(columns={'provincia_destino': 'provincia', 'valor': 'entradas'})
)

df_movimientos = pd.merge(entradas, salidas, on=['provincia', 'anyo'], how='outer')
df_movimientos['entradas']   = df_movimientos['entradas'].fillna(0).astype(int)
df_movimientos['salidas']    = df_movimientos['salidas'].fillna(0).astype(int)
df_movimientos['saldo_neto'] = df_movimientos['entradas'] - df_movimientos['salidas']
df_movimientos = df_movimientos.sort_values(['provincia', 'anyo']).reset_index(drop=True)

print(f"Movimientos provincia×año → {df_movimientos.shape}")
print(f"Provincias: {df_movimientos['provincia'].nunique()}")
print()
print(df_movimientos[df_movimientos['provincia'] == 'Madrid'].to_string(index=False))


Movimientos provincia×año → (742, 5)
Provincias: 53

provincia  anyo  entradas  salidas  saldo_neto
   Madrid  2008         0    79288      -79288
   Madrid  2009         0    73490      -73490
   Madrid  2010         0    72549      -72549
   Madrid  2011         0    67771      -67771
   Madrid  2012         0    62397      -62397
   Madrid  2013         0    59207      -59207
   Madrid  2014         0    57410      -57410
   Madrid  2015         0    55077      -55077
   Madrid  2016         0    55188      -55188
   Madrid  2017         0    54993      -54993
   Madrid  2018         0    62471      -62471
   Madrid  2019         0    67176      -67176
   Madrid  2020         0    70288      -70288
   Madrid  2021         0    78600      -78600


In [4]:
df_saldo_clean = df_saldo[
    (df_saldo['sexo']       == 'Total') &
    (df_saldo['grupo_edad'] == 'Todas las edades') &
    df_saldo['provincia'].notna() &
    df_saldo['valor'].notna()
][['provincia', 'anyo', 'valor']].copy()

df_saldo_clean = df_saldo_clean.rename(columns={'valor': 'saldo_ine'})
df_saldo_clean = df_saldo_clean.sort_values(['provincia', 'anyo']).reset_index(drop=True)

print(f"Saldo INE provincia×año → {df_saldo_clean.shape}")
print(f"Provincias: {df_saldo_clean['provincia'].nunique()}")
print()
print(df_saldo_clean[df_saldo_clean['provincia'] == 'Madrid'].to_string(index=False))


Saldo INE provincia×año → (728, 3)
Provincias: 1

Empty DataFrame
Columns: [provincia, anyo, saldo_ine]
Index: []


In [5]:
df_evr = pd.merge(df_movimientos, df_saldo_clean, on=['provincia', 'anyo'], how='left')

# Validación: diferencia entre saldo calculado y saldo oficial INE
df_evr['diff_saldo'] = df_evr['saldo_neto'] - df_evr['saldo_ine']
max_diff = df_evr['diff_saldo'].abs().max()

print(f"Shape tabla maestra EVR: {df_evr.shape}")
print(f"Diferencia máx saldo calculado vs INE: {max_diff:.0f} personas (esperado ≤ 10 por redondeo)")
print(f"Nulos:\n{df_evr.isnull().sum()}")
print()
print(df_evr[df_evr['provincia'] == 'Madrid'].to_string(index=False))


Shape tabla maestra EVR: (742, 7)
Diferencia máx saldo calculado vs INE: nan personas (esperado ≤ 10 por redondeo)
Nulos:
provincia       0
anyo            0
entradas        0
salidas         0
saldo_neto      0
saldo_ine     742
diff_saldo    742
dtype: int64

provincia  anyo  entradas  salidas  saldo_neto  saldo_ine  diff_saldo
   Madrid  2008         0    79288      -79288        NaN         NaN
   Madrid  2009         0    73490      -73490        NaN         NaN
   Madrid  2010         0    72549      -72549        NaN         NaN
   Madrid  2011         0    67771      -67771        NaN         NaN
   Madrid  2012         0    62397      -62397        NaN         NaN
   Madrid  2013         0    59207      -59207        NaN         NaN
   Madrid  2014         0    57410      -57410        NaN         NaN
   Madrid  2015         0    55077      -55077        NaN         NaN
   Madrid  2016         0    55188      -55188        NaN         NaN
   Madrid  2017         0    54993    

In [6]:
print(f"{'Dataset':<40} {'Filas':>6}  {'Cols':>4}  {'Nulos%':>6}  {'Años'}")
print("-" * 70)

for nombre, df in [
    ("flujo_interprovincial_limpio",  df_flujo_clean),
    ("movimientos_provincia_anyo",    df_movimientos),
    ("saldo_ine_provincia_anyo",      df_saldo_clean),
    ("evr_maestra_provincia_anyo",    df_evr),
]:
    filas = len(df)
    cols  = len(df.columns)
    nulos = round(df.isnull().sum().sum() / (filas * cols) * 100, 1) if filas else 0
    anyo_min = int(df['anyo'].min())
    anyo_max = int(df['anyo'].max())
    print(f"{nombre:<40} {filas:>6}  {cols:>4}  {nulos:>5}%  {anyo_min}–{anyo_max}")


Dataset                                   Filas  Cols  Nulos%  Años
----------------------------------------------------------------------
flujo_interprovincial_limpio               2184     6    0.0%  2008–2021
movimientos_provincia_anyo                  742     5    0.0%  2008–2021
saldo_ine_provincia_anyo                    728     3    0.0%  2008–2021
evr_maestra_provincia_anyo                  742     7   28.6%  2008–2021


In [7]:
df_evr_final = df_evr.drop(columns=['diff_saldo'])

outputs = {
    "clean_evr_provincia_anio.parquet":      df_evr_final,
    "clean_flujo_interprovincial.parquet":   df_flujo_clean,
    "clean_saldo_ine_provincia_anio.parquet": df_saldo_clean,
}

for fname, df in outputs.items():
    path = OUTPUT_DIR / fname
    df.to_parquet(path, index=False)
    print(f"✅ Guardado: {path} ({len(df)} filas)")

print("\n🎉 Todos los parquets EVR silver guardados en output/evr/02-silver")


✅ Guardado: ../../output/evr/02-silver/clean_evr_provincia_anio.parquet (742 filas)
✅ Guardado: ../../output/evr/02-silver/clean_flujo_interprovincial.parquet (2184 filas)
✅ Guardado: ../../output/evr/02-silver/clean_saldo_ine_provincia_anio.parquet (728 filas)

🎉 Todos los parquets EVR silver guardados en output/evr/02-silver
